# Q5 - PyTorch on CIFAR-10

In [ ]:
import os, sys
if os.path.exists('/kaggle/working/ML_Assignment'):
    os.system('cd /kaggle/working/ML_Assignment && git pull')
else:
    os.system('git clone https://github.com/AvtandilSh1/ML_Assignment.git /kaggle/working/ML_Assignment')
sys.path.insert(0, '/kaggle/working/ML_Assignment')
os.chdir('/kaggle/working/ML_Assignment')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torchvision.datasets as dset
import torchvision.transforms as T
import numpy as np

dtype = torch.float32
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
NUM_TRAIN = 49000
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

cifar10_train = dset.CIFAR10('./cs231n/datasets', train=True, download=True, transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./cs231n/datasets', train=True, download=True, transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./cs231n/datasets', train=False, download=True, transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

In [ ]:
def flatten(x):
    return x.view(x.shape[0], -1)

def random_weight(shape):
    fan_in = shape[0] if len(shape) == 2 else np.prod(shape[1:])
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

def get_val_acc(loader, model_fn, params=None):
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params) if params is not None else model_fn(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
    return float(num_correct) / num_samples

## Part II - Barebones PyTorch

In [ ]:
def three_layer_convnet(x, params):
    conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b = params
    x = F.relu(F.conv2d(x, conv_w1, conv_b1, padding=2))
    x = F.relu(F.conv2d(x, conv_w2, conv_b2, padding=1))
    x = flatten(x)
    return x.mm(fc_w) + fc_b

x = torch.zeros((64, 3, 32, 32), dtype=dtype)
conv_w1 = torch.zeros((6, 3, 5, 5), dtype=dtype)
conv_b1 = torch.zeros(6)
conv_w2 = torch.zeros((9, 6, 3, 3), dtype=dtype)
conv_b2 = torch.zeros(9)
fc_w = torch.zeros((9 * 32 * 32, 10))
fc_b = torch.zeros(10)
scores = three_layer_convnet(x, [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b])
print('three_layer_convnet output shape (expect [64,10]):', scores.size())

In [ ]:
channel_1, channel_2 = 32, 16
conv_w1 = random_weight((channel_1, 3, 5, 5))
conv_b1 = zero_weight((channel_1,))
conv_w2 = random_weight((channel_2, channel_1, 3, 3))
conv_b2 = zero_weight((channel_2,))
fc_w    = random_weight((channel_2 * 32 * 32, 10))
fc_b    = zero_weight((10,))
params  = [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b]

for t, (x, y) in enumerate(loader_train):
    x = x.to(device=device, dtype=dtype)
    y = y.to(device=device, dtype=torch.long)
    scores = three_layer_convnet(x, params)
    loss = F.cross_entropy(scores, y)
    loss.backward()
    with torch.no_grad():
        for w in params:
            w -= 3e-3 * w.grad
            w.grad.zero_()

acc = get_val_acc(loader_val, three_layer_convnet, params)
print(f'Part II ConvNet val acc: {acc:.4f}')

## Part III - Module API

In [ ]:
class ThreeLayerConvNet(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channel, channel_1, 5, padding=2)
        nn.init.kaiming_normal_(self.conv1.weight)
        self.conv2 = nn.Conv2d(channel_1, channel_2, 3, padding=1)
        nn.init.kaiming_normal_(self.conv2.weight)
        self.fc = nn.Linear(channel_2 * 32 * 32, num_classes)
        nn.init.kaiming_normal_(self.fc.weight)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.fc(flatten(x))

x = torch.zeros((64, 3, 32, 32), dtype=dtype)
model = ThreeLayerConvNet(3, 12, 8, 10)
print('ThreeLayerConvNet output shape (expect [64,10]):', model(x).size())

In [ ]:
model = ThreeLayerConvNet(in_channel=3, channel_1=32, channel_2=16, num_classes=10).to(device)
optimizer = optim.SGD(model.parameters(), lr=3e-3)

model.train()
for x, y in loader_train:
    x = x.to(device=device, dtype=dtype)
    y = y.to(device=device, dtype=torch.long)
    optimizer.zero_grad()
    F.cross_entropy(model(x), y).backward()
    optimizer.step()

model.eval()
acc = get_val_acc(loader_val, model)
print(f'Part III val acc: {acc:.4f}')

## Part IV - Sequential API

In [ ]:
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

channel_1, channel_2 = 32, 16
model = nn.Sequential(
    nn.Conv2d(3, channel_1, 5, padding=2), nn.ReLU(),
    nn.Conv2d(channel_1, channel_2, 3, padding=1), nn.ReLU(),
    Flatten(),
    nn.Linear(channel_2 * 32 * 32, 10)
).to(device)
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True)

model.train()
for x, y in loader_train:
    x = x.to(device=device, dtype=dtype)
    y = y.to(device=device, dtype=torch.long)
    optimizer.zero_grad()
    F.cross_entropy(model(x), y).backward()
    optimizer.step()

model.eval()
acc = get_val_acc(loader_val, model)
print(f'Part IV val acc: {acc:.4f}')

## Part V - Open Challenge (>=70%)

In [ ]:
transform_aug = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
cifar10_aug = dset.CIFAR10('./cs231n/datasets', train=True, download=False, transform=transform_aug)
loader_aug = DataLoader(cifar10_aug, batch_size=128,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)), num_workers=2)

In [ ]:
class BestModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),   nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, 3, padding=1),  nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

In [ ]:
best_model = BestModel().to(device)
optimizer = optim.Adam(best_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

for epoch in range(10):
    best_model.train()
    total_loss = 0
    for x, y in loader_aug:
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)
        optimizer.zero_grad()
        loss = F.cross_entropy(best_model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    best_model.eval()
    val_acc = get_val_acc(loader_val, best_model)
    print(f'Epoch {epoch+1:2d} | loss: {total_loss/len(loader_aug):.4f} | val_acc: {val_acc:.4f}')

best_model.eval()
test_acc = get_val_acc(loader_test, best_model)
print(f'Test acc: {test_acc:.4f}')

## MLflow - DagsHub-ზე Logging

In [ ]:
!pip install -q mlflow
import mlflow, logging, contextlib, io
logging.getLogger('mlflow').setLevel(logging.ERROR)
os.environ['MLFLOW_TRACKING_USERNAME'] = 'ashos22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '78b25562699413e85d83414b742443e9df7c5cb5'

with contextlib.redirect_stdout(io.StringIO()):
    mlflow.set_tracking_uri('https://dagshub.com/ashos22/ML_Assignment.mlflow')
    mlflow.set_experiment('Q5-PyTorch-CIFAR10')
    with mlflow.start_run(run_name='best-model-vgg-like'):
        mlflow.log_params({
            'architecture': 'VGG-like with BN',
            'optimizer': 'Adam',
            'epochs': 10,
            'batch_size': 128,
            'data_augmentation': 'RandomCrop+HFlip'
        })
        mlflow.log_metrics({
            'val_acc': float(val_acc),
            'test_acc': float(test_acc)
        })

print('Logged to DagsHub.')